# Notebook 02 — Traditional RAG (Baseline)

## Learning Goals
- Build a complete Traditional RAG pipeline
- Ask 5 test questions and see retrieved chunks
- Understand WHY traditional RAG sometimes fails
- Save results for comparison in Notebook 06

## How Traditional RAG Works
```
Query → Embed → Search FAISS → Get top-k chunks → LLM → Answer
```
No context is added to chunks — they are stored as-is.

In [ ]:
import os
import sys
import json
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from utils.pdf_extractor import extract_full_text, get_paper_metadata
from utils.chunker import fixed_size_chunk
from utils.embedder import load_embedding_model, embed_chunks, build_faiss_index, save_index
from utils.retriever import retrieve_top_k, format_context, display_results
from utils.llm import generate_answer
import importlib.util

def load_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

traditional = load_module('traditional', '../context_builders/traditional.py')

print('✅ All imports ready!')

## Step 1: Load and Chunk the Paper

In [ ]:
# Load paper
meta = get_paper_metadata('../data/paper.pdf')
paper_title = meta['title']
print(f'Paper: {paper_title[:60]}...')

# Extract and chunk
full_text = extract_full_text('../data/paper.pdf')
raw_chunks = fixed_size_chunk(full_text, chunk_size=500, chunk_overlap=50)
print(f'Total chunks: {len(raw_chunks)}')

## Step 2: Build Traditional RAG Index
No context added — chunks stored as-is.

In [ ]:
# Apply traditional context builder (no change)
chunks = traditional.build(raw_chunks)

# Load embedding model
embedding_model = load_embedding_model()

# Embed chunks
embeddings = embed_chunks(chunks, embedding_model)

# Build FAISS index
index = build_faiss_index(embeddings)

# Save for later use
save_index(index, chunks, '../outputs', 'traditional')

print('\n✅ Traditional RAG index ready!')
print(f'   Chunks: {len(chunks)}')
print(f'   Index saved to outputs/traditional_index.faiss')

## Step 3: Define Test Questions

These 5 questions test different aspects:
- **Q1**: High-level concept question
- **Q2**: Technical/structural question
- **Q3**: Experimental details question
- **Q4**: Comparison question
- **Q5**: Limitation question

In [ ]:
test_questions = [
    'What problem does MGranRAG solve?',
    'How is the Contextual Hierarchical Graph constructed?',
    'What datasets were used in the experiments?',
    'How does MGranRAG compare to HippoRAG 2?',
    'What are the limitations of MGranRAG?'
]

print('Test Questions:')
for i, q in enumerate(test_questions):
    print(f'  Q{i+1}: {q}')

## Step 4: Ask Questions and See Results

For each question we show:
1. Retrieved chunks (what the system found)
2. The answer generated by LLM
3. Quality observation

In [ ]:
# Store results for comparison later
traditional_results = []

for i, question in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f'Q{i+1}: {question}')
    print(f"{'='*60}")

    # Retrieve top-3 chunks
    retrieved = retrieve_top_k(
        query=question,
        index=index,
        chunks=chunks,
        model=embedding_model,
        k=3
    )

    # Show retrieved chunks
    print(f'\n📦 Retrieved Chunks:')
    for j, chunk in enumerate(retrieved):
        print(f'\n  Chunk {j+1} (score: {chunk["retrieval_score"]:.3f})')
        print(f'  Section: {chunk.get("section", "UNKNOWN")}')
        print(f'  Text: {chunk["text"][:200]}...')

    # Format context and generate answer
    context = format_context(retrieved)
    answer = generate_answer(question, context)

    print(f'\n💬 Answer:')
    print(f'  {answer}')

    # Save result
    traditional_results.append({
        'question': question,
        'retrieved_chunks': [
            {
                'chunk_id': c['chunk_id'],
                'score': c['retrieval_score'],
                'text': c['text'][:300],
                'section': c.get('section', 'Unknown')
            }
            for c in retrieved
        ],
        'answer': answer,
        'rag_type': 'traditional'
    })

print(f"\n{'='*60}")
print(f'✅ All {len(test_questions)} questions answered!')

## Step 5: Observe the Problems

Look at the retrieved chunks above and notice:
- Do they actually answer the question?
- Do they contain the right section info?
- Are numbers/tables being retrieved instead of text?

These are the problems Traditional RAG has!

In [ ]:
print('TRADITIONAL RAG PROBLEM ANALYSIS')
print('=' * 60)

problems_found = 0

for i, result in enumerate(traditional_results):
    question = result['question']
    chunks_retrieved = result['retrieved_chunks']

    print(f"\nQ{i+1}: {question}")

    # Check if chunks have section info
    has_section = any(
        c['section'] not in ['Unknown', 'N/A', '']
        for c in chunks_retrieved
    )

    # Check if chunks contain numbers/tables
    has_numbers = any(
        sum(c.isdigit() for c in chunk['text']) > 50
        for chunk in chunks_retrieved
    )

    if not has_section:
        print(f'  ⚠️  No section info in retrieved chunks')
        problems_found += 1

    if has_numbers:
        print(f'  ⚠️  Retrieved chunks contain mostly numbers/tables')
        problems_found += 1

    avg_score = sum(c['score'] for c in chunks_retrieved) / len(chunks_retrieved)
    print(f'  📊 Avg retrieval score: {avg_score:.3f}')

    if avg_score < 0.3:
        print(f'  ⚠️  Low retrieval scores — chunks may not be relevant')
        problems_found += 1

print(f"\n{'='*60}")
print(f'Total problems found: {problems_found}')
print(f'This is why we need Contextual RAG!')

In [ ]:
# Save results for Notebook 06 comparison
with open('../outputs/results_traditional.json', 'w') as f:
    json.dump(traditional_results, f, indent=2)

print('✅ Results saved to outputs/results_traditional.json')
print(f'✅ {len(traditional_results)} questions answered')
print('\nNext: Notebook 03 — Window RAG')

## Summary

**Traditional RAG Problems:**
1. Chunks have NO section information
2. Math/table chunks retrieved instead of text
3. Low retrieval scores on complex questions
4. LLM gets poor context → poor answers

**Next notebook:** Window RAG fixes the surrounding context problem!